In [1]:
import sys
import os
import pandas as pd
import astropy.units as u

from pathlib import Path
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS

parent_dir = os.path.abspath(os.path.join(os.path.dirname('utils.py'), ".."))
sys.path.append(parent_dir)
import utils as ut

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
CUT = True
MASK = False

In [4]:
cwd = Path.cwd()
parent = cwd.parent
table_dir = parent / 'train val test tables'
mosaic_dir = parent / 'data' / 'mosaics'
cutout_dir = parent / 'data' / 'cutouts'
masked = parent / 'data' / 'masked_cuts'

In [5]:
# get all train val test tables
table_dfs = []
for table in table_dir.rglob('*'):
    table_df = pd.read_csv(table, index_col=0)
    table_df['table'] = table.stem
    table_dfs.append(table_df)

# combine them
df = pd.concat(table_dfs)
df = df[['RA', 'DEC', 'excess', 'flag', 'survey', 'table']]

print("Num gals (expect 486): ", len(df))

C:\Users\harty\AppData\Local\Temp\ipykernel_11888\3522837558.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  table_df['table'] = table.stem
C:\Users\harty\AppData\Local\Temp\ipykernel_11888\3522837558.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  table_df['table'] = table.stem
C:\Users\harty\AppData\Local\Temp\ipykernel_11888\3522837558.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at 

Num gals (expect 486):  486


C:\Users\harty\AppData\Local\Temp\ipykernel_11888\3522837558.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  table_df['table'] = table.stem


In [6]:
# get mosaic headers
headers = ut.fits_df(mosaic_dir)
headers = headers[['FILENAME'] + [col for col in headers.columns if col != 'FILENAME']]

In [7]:
if CUT: 
    with_two = []
    size = 2

    stamp = u.Quantity([size, size], u.arcsec)

    # loop through each mosaic
    for _, row in headers.iterrows():
        path = row['FILEPATH']
        catalogs = list(df['survey'].unique())
        bands = ['f150w', 'f277w', 'f444w']

        band = None
        catalog = None

        # determine the band and survey of the current mosaic
        for c, b in zip(catalogs, bands):
            if c in path.name:
                catalog = c
            if b in path.name:
                band = b

        # load mosiac data
        hdul = fits.open(path)
        data = hdul[0].data
        wcs = WCS(hdul[0].header)

        # loop through df filtered for that catalog
        survey_df = df[df['survey'] == catalog]
        for i, row in survey_df.iterrows():

            arcsecs = ((size/2) * u.arcsec)
            ra = row['RA'] * u.deg
            dec = row['DEC'] * u.deg

            pos = SkyCoord(ra, dec, frame="icrs")
            cutout = Cutout2D(data, pos, stamp, wcs=wcs)

            header = fits.Header()
            for col_name, value in row.items():
                if pd.isna(value):
                    continue
                key = str(col_name)[:8].upper()
                header[key] = value
            header['BAND'] = band

            folder = Path(cutout_dir / f"{float(i)}")
            folder.mkdir(parents=True, exist_ok=True)

            if MASK:
                cutout.data = ut.mask_other_sources(cutout.data, nsigma=3)[0]

            cutout_fits = fits.PrimaryHDU(data=cutout.data, header=header)
            cutout_fits.writeto(folder / f"{band}.fits", overwrite=True)

        hdul.close()

    size = len(list(cutout_dir.glob('*')))
    print(f"Number of cutouts: {size}")
    print("Expected: 486")

    num_bands = 3

    for folder in cutout_dir.iterdir():
        if folder.is_dir():
            count = len(list(folder.glob('*')))
            if count != num_bands:
                print(f"{folder.name}")

Set DATE-AVG to '2022-09-21T10:17:11.746' from MJD-AVG.
Set DATE-END to '2022-12-22T05:42:36.326' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -36.758463 from OBSGEO-[XYZ].
Set OBSGEO-H to 1725319218.494 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2022-09-22T03:18:18.036' from MJD-AVG.
Set DATE-END to '2022-12-25T03:57:18.887' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -36.739053 from OBSGEO-[XYZ].
Set OBSGEO-H to 1725216481.136 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2022-10-07T06:02:04.794' from MJD-AVG.
Set DATE-END to '2022-12-22T06:44:09.762' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -36.768884 from OBSGEO-[XYZ].
Set OBSGEO-H to 1725373838.960 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Number of cutouts: 486
Expected: 486
